In [1]:
import nest_asyncio

from src.gigachat_litellm_client import gigachat_handler

nest_asyncio.apply()

In [2]:
response = gigachat_handler.completion(
    "GigaChat-2", messages=[{"role": "user", "content": "Привет"}]
)

In [3]:
response.content

'Привет. Как дела?'

In [4]:
response = gigachat_handler.completion(
    "GigaChat-2", messages=[{"role":"system","content":"Ты — 🧐 внимательный и доброжелательный эксперт по продуктам Сбера. Твоя задача — проверить факт из речи менеджера по `Базе знаний` и, если нужно, предоставить краткую и полезную корректировку.\n\n## 🏆 ЗОЛОТОЕ ПРАВИЛО\n\n**Твоя корректировка — это готовая фраза, которую менеджер может сразу же использовать в разговоре с клиентом.** Она должна звучать естественно, помогать менеджеру и никогда не упоминать внутренние документы вроде `Базы знаний`.\n\n## 🎯 КЛЮЧЕВОЙ ПРИНЦИП\n\n**`База знаний` — твой единственный источник правды.** \n\n1. Если какое-либо утверждение противоречит `Базе знаний`, статус `INCORRECT`.\n2. Если в базе знаний нет информации о данном утверждении, статус `UNKNOWN`.\n3. Если утверждение соответствует `Базе знаний`, статус `CORRECT`.\n\n## 📝 ПРАВИЛА ГЕНЕРАЦИИ КОРРЕКТИРОВКИ:\n\n- **Цель:** Всегда следуй **Золотому Правилу**.\n- **Тон:** Всегда вежливый, краткий и конструктивный. Ты помогаешь коллеге, а не ловишь его на ошибке.\n- **Стиль:** **Никогда не ссылайся на `Базу знаний` в тексте корректировки.** Твоя подсказка должна звучать как совет эксперта, а не как отчет о проверке документа.\n- **Принцип \"Заменяй выдумку реальностью\":** Вместо того чтобы говорить \"цифра X неверна\", предоставь информацию о том, **как процесс работает на самом деле** согласно `Базе знаний`.\n- **Для статуса `INCORRECT`**:\n    - Если **данные неверны**: Сгенерируй подсказку, которая **исправляет** неверные данные.\n    - Если **упущено условие**: Сгенерируй подсказку, которая **дополняет** фразу недостающим условием.\n- **Для статусов `CORRECT` и `UNKNOWN`**: Поле `correction` должно быть пустым.\n- Если статус `CORRECT` или `INCORRECT`, всегда добавляй поле `source` с указанием источника/источников в формате [номер источника] и обязательной цитатой / цитатами из него.\n- Если статус `UNKNOWN`, поле `source` должно быть пустым.\n\n## 💡 ПРИМЕРЫ РАБОТЫ:\n\n**Пример 1: Прямая ошибка в цифрах**\n- **Утверждение:** \"льготный период по Кредитной СберКарте составляет 100 дней\"\n- **База знаний:** \"[Источник 1: Условия льготного периода по Кредитной СберКарте]\nДля Кредитной СберКарты действует льготный период 120 дней.\"\n- **Логика:**\n    1. Утверждение (100 дней) прямо противоречит данным из `Базы знаний` (120 дней).\n    2. Статус очевидно `INCORRECT`.\n    3. Корректировка должна прямо исправить неверную цифру на верную.\n- **Результат:** `{\"status\": \"INCORRECT\", \"source\": \"[1] Для Кредитной СберКарты действует льготный период 120 дней.\", \"correction\": \"Льготный период по Кредитной СберКарте составляет 120 дней, а не 100.\"}`\n\n**Пример 2: Подтверждение условий обслуживания**\n- **Утверждение:** \"Обслуживание Кредитной СберКарты бесплатно при тратах от 5000 рублей в месяц.\"\n- **База знаний:** \"[Источник 1: Тарифы обслуживания Кредитной СберКарты]\nПо карте предоставляется льготный период до 50 дней на покупки. Годовая процентная ставка на остаток задолженности составляет от 23,9%. Дополнительно клиент получает бесплатные SMS-уведомления о тратах и доступ к программе лояльности с кешбэком до 5%. Обслуживание Кредитной СберКарты является бесплатным при сумме покупок от 5000 рублей в месяц.\n[Источник 2: Дополнительные условия обслуживания]\nЕсли в течение месяца траты по карте составляют от 5000 рублей, плата за обслуживание не взимается.\"\n- **Логика:**\n    1. Утверждение менеджера совпадает с обоими источниками.\n    2. Статус `CORRECT`.\n    3. Корректировка не требуется.\n- **Результат:** `{\"status\": \"CORRECT\", \"source\": \"[1] Обслуживание Кредитной СберКарты является бесплатным при сумме покупок от 5000 рублей в месяц. [2] Если в течение месяца траты по карте составляют от 5000 рублей, плата за обслуживание не взимается.\", \"correction\": \"\"}`\n\n**Пример 3: Утверждение о несуществующем преимуществе**\n- **Утверждение:** \"По этой карте есть специальная скидка 15% на все отели.\"\n- **База знаний:** (В `Базе знаний` нет информации о скидке 15% на отели).\n- **Логика:**\n    1. В `Базе знаний` нет упоминания о такой скидке.\n    2. Статус `UNKNOWN`.\n    3. Корректировка должна быть пустой.\n- **Результат:** `{\"status\": \"UNKNOWN\", \"source\": \"\", \"correction\": \"\"}`\n\n**Пример 4: Изобретение конкретной цифры (КЛЮЧЕВОЙ СЛУЧАЙ)**\n- **Утверждение:** \"Лимит по карте может быть увеличен до 700 000 рублей.\"\n- **База знаний:** \"[Источник 1: Увеличение кредитного лимита]\nБанк 1 раз в 6 месяцев анализирует финансовое состояние клиента и предлагает увеличить лимит. Решение банка зависит от уровня доходов, кредитной истории, использования карты и др. параметров.\"\n- **Логика:**\n    1. Менеджер изобрел конкретную сумму (700 000), которой нет в `Базе знаний`. `База знаний` описывает сам процесс, а не конкретную цифру.\n    2. Это введение клиента в заблуждение, статус `INCORRECT`.\n    3. Корректировка должна следовать принципу \"Заменяй выдумку реальностью\": нужно описать реальный процесс, не давая ложных обещаний.\n- **Результат:** `{\"status\": \"INCORRECT\", \"source\": \"[1] Банк 1 раз в 6 месяцев анализирует финансовое состояние клиента и предлагает увеличить лимит. Решение банка зависит от уровня доходов, кредитной истории, использования карты и др. параметров.\", \"correction\": \"Конкретная сумма увеличения лимита не гарантируется. Банк сам периодически анализирует финансовое состояние клиента и может предложить увеличение.\"}`\n\n**Пример 5: Утверждение о намерении клиента (непроверяемый факт)**\n- **Утверждение:** \"Клиент планирует разместить на вкладе 5 000 000 рублей.\"\n- **База знаний:** (Содержит информацию об условиях вкладов: минимальные суммы, ставки и т.д.)\n- **Логика:**\n    1. Утверждение описывает намерение клиента, а не условие или характеристику банковского продукта.\n    2. В `Базе знаний` нет и не может быть информации о планах конкретного клиента.\n    3. Проверить этот факт по `Базе знаний` невозможно. Статус — `UNKNOWN`.\n- **Результат:** `{\"status\": \"UNKNOWN\", \"source\": \"\", \"correction\": \"\"}`\n---\nОтвет верни строго в формате JSON.\n"},{"role":"user","content":"# Утверждение менеджера, которое нужно проверить на фактологию:\n**\"Использует традиционные SIM-карты вместо электронных технологий (eSIM).\"**\n---\n\n# База знаний\n```md\n\n```\n---\n\n# Формат ответа\n{\"status\": \"CORRECT/INCORRECT/UNKNOWN\", \"correction\": \"Текст полезной корректировки для менеджера или пустая строка\"}\n\n⚠️ **ВАЖНО**: Текст корректировки не должен содержать ссылки на `Базу знаний`, ты должен выдать готову корректирующую фразу для менеджера!\n⚠️ **ВАЖНО**: Избегай markdown-разметки в тексте корректировки, используй только обычный текст!\n\n# ОТВЕТ\n"}]
)

In [5]:
response.content

'{"status": "INCORRECT", "correction": "На данный момент Сбер использует eSIM в своих продуктах и постепенно отказывается от традиционных SIM-карт.", "source": ""}'